# Final Thesis Results Dashboard

**Research question:** Does representation stabilization imply representation sufficiency?

---

Run all cells top-to-bottom (`Kernel > Restart & Run All`).
The final cell renders the full scrollable dashboard inline in the notebook.

To export HTML: change the last line of the final cell to `build_dashboard(save_html=True)`.
Output goes to `notebooks/.generated/dashboard.html` (git-ignored, never committed).

| Cell | Purpose |
|------|---------|
| 1 | Imports + paths |
| 2 | File discovery |
| 3 | Load & inspect CSVs |
| 4 | Cross-LR matplotlib comparison figures |
| 5 | Saturation analysis tables |
| 6 | Thesis hypothesis summary table |
| 7 | `build_dashboard()` — full inline HTML dashboard |


In [ ]:
import os, json, base64, io, warnings
import pandas as pd
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
from IPython.display import HTML, display
warnings.filterwarnings("ignore")

matplotlib.rcParams["figure.dpi"] = 110
matplotlib.rcParams["axes.grid"] = True
matplotlib.rcParams["grid.alpha"] = 0.3
matplotlib.rcParams["axes.spines.top"] = False
matplotlib.rcParams["axes.spines.right"] = False

try:
    NB_DIR = os.path.dirname(os.path.abspath(__file__))
except NameError:
    NB_DIR = os.getcwd()   # Jupyter: cwd is typically the notebook directory
BASE = os.path.normpath(os.path.join(NB_DIR, "..", "thesis_results"))
GEN_DIR = os.path.join(NB_DIR, ".generated")
os.makedirs(GEN_DIR, exist_ok=True)

RUN_DIRS = {
    "0.05": "final_lr005_seed42",
    "0.10": "final_lr010_seed42",
    "0.20": "final_lr020_seed42",
}
CSV_NAMES = [
    "master_trajectory.csv",
    "cka_summary.csv",
    "neural_collapse.csv",
    "entk_summary.csv",
    "probe_results_wide.csv",
    "probe_results_long.csv",
]
LR_COLORS = {"0.05": "#1f77b4", "0.10": "#ff7f0e", "0.20": "#2ca02c"}
print("Setup complete.  BASE =", BASE)


In [ ]:
# Step 1: File discovery
runs = {}
for lr, run_dir in RUN_DIRS.items():
    run_path = os.path.join(BASE, run_dir)
    info = {
        "lr": lr, "dir": run_dir, "path": run_path,
        "exists": os.path.isdir(run_path),
        "csvs": {}, "plots": [], "jsons": [],
    }
    if info["exists"]:
        res_dir   = os.path.join(run_path, "results")
        plots_dir = os.path.join(run_path, "plots")
        if os.path.isdir(res_dir):
            for f in sorted(os.listdir(res_dir)):
                fp = os.path.join(res_dir, f)
                if f.endswith(".csv"):   info["csvs"][f] = fp
                elif f.endswith(".json"):info["jsons"].append(fp)
        if os.path.isdir(plots_dir):
            for f in sorted(os.listdir(plots_dir)):
                if f.endswith(".png"):
                    info["plots"].append(os.path.join(plots_dir, f))
    runs[lr] = info

for lr, info in runs.items():
    status = "OK" if info["exists"] else "MISSING"
    print(f"LR={lr}  [{status}]  dir={info['dir']}")
    print(f"  CSVs : {list(info['csvs'].keys())}")
    print(f"  PNGs : {[os.path.basename(p) for p in info['plots']]}")
    print(f"  JSONs: {[os.path.basename(p) for p in info['jsons']]}")
    print()


In [ ]:
# Step 2: Load and inspect all CSVs
dfs = {}
for lr, info in runs.items():
    dfs[lr] = {}
    for csv_name in CSV_NAMES:
        path = info["csvs"].get(csv_name)
        if path and os.path.exists(path):
            df = pd.read_csv(path)
            dfs[lr][csv_name] = df
            miss = {c: int(n) for c, n in df.isnull().sum().items() if n > 0}
            print(f"[LR={lr}] {csv_name}  shape={df.shape}")
            print(f"  columns : {list(df.columns)}")
            print(f"  first 3 rows:")
            print(df.head(3).to_string(index=False))
            print(f"  final row:")
            print(df.tail(1).to_string(index=False))
            if miss:
                print(f"  missing : {miss}")
            print()
        else:
            print(f"[LR={lr}] {csv_name}: MISSING")


In [ ]:
# Step 6: Cross-LR comparison figures (matplotlib)
def _col(lr, csv, col):
    df = dfs.get(lr, {}).get(csv)
    if df is None or col not in df.columns: return None, None
    return df["epoch"], df[col]

PANELS = [
    ("Test Accuracy (network)",   "master_trajectory.csv", "network_test_acc"),
    ("Logistic Regression Acc",   "master_trajectory.csv", "logistic_acc"),
    ("Linear SVM Acc",            "master_trajectory.csv", "linear_svm_acc"),
    ("RBF SVM Acc",               "master_trajectory.csv", "rbf_svm_acc"),
    ("Random Forest Acc",         "master_trajectory.csv", "rf_acc"),
    ("CKA to Final",              "master_trajectory.csv", "cka_to_final"),
    ("Mean Future CKA",           "master_trajectory.csv", "mean_future_cka"),
    ("Local CKA Change",          "master_trajectory.csv", "local_cka_change"),
    ("log10(NC1)",                "master_trajectory.csv", "log10_nc1"),
    ("NC2 ETF Deviation",         "master_trajectory.csv", "nc2_etf_deviation"),
    ("NC3 Weight-Mean Alignment", "master_trajectory.csv", "nc3_weight_mean_alignment"),
    ("NC4 NCM Disagreement",      "master_trajectory.csv", "nc4_ncm_disagreement"),
    ("eNTK dist to final",        "master_trajectory.csv", "entk_distance_final"),
    ("Mean Future eNTK Sim",      "master_trajectory.csv", "mean_future_entk_similarity"),
    ("Learning Rate",             "master_trajectory.csv", "lr"),
]

active = [(t, c, col) for t, c, col in PANELS
          if any(_col(lr, c, col)[0] is not None for lr in RUN_DIRS)]

if not active:
    print("No data available for cross-LR comparison plots.")
else:
    ncols  = 3
    nrows  = (len(active) + ncols - 1) // ncols
    fig, axes = plt.subplots(nrows, ncols, figsize=(18, nrows * 3.5))
    axes = axes.flatten() if nrows > 1 else [axes] * ncols

    for i, (title, csv, col) in enumerate(active):
        ax = axes[i]
        for lr in RUN_DIRS:
            ep, vals = _col(lr, csv, col)
            if ep is not None:
                ax.plot(ep, vals, label=f"lr={lr}", color=LR_COLORS[lr], linewidth=1.6)
        ax.set_title(title, fontsize=10, fontweight="bold")
        ax.set_xlabel("Epoch", fontsize=8)
        ax.legend(fontsize=7)
        ax.tick_params(labelsize=7)

    for i in range(len(active), len(axes)):
        axes[i].set_visible(False)

    fig.suptitle("Cross-LR Comparison — All Metrics", fontsize=14, fontweight="bold", y=1.01)
    plt.tight_layout()
    plt.show()
    print(f"Rendered {len(active)} panels.")


In [ ]:
# Step 5: Saturation analysis
def _first_epoch(df, col, thr, direction):
    if col not in df.columns: return None
    for ep, v in zip(df["epoch"], df[col]):
        if pd.isna(v): continue
        if direction == "above" and v >= thr: return int(ep)
        if direction == "below" and v <= thr: return int(ep)
    return None

PROBE_COLS   = ["logistic_acc", "linear_svm_acc", "rbf_svm_acc", "rf_acc"]
PROBE_LABELS = ["Logistic", "LinearSVM", "RBFSVM", "RandForest"]

rows = []
for lr in RUN_DIRS:
    df = dfs.get(lr, {}).get("master_trajectory.csv")
    if df is None: continue
    row = {"LR": lr}
    for pcol, plabel in zip(PROBE_COLS, PROBE_LABELS):
        if pcol not in df.columns: continue
        series = df[pcol].dropna()
        if series.empty: continue
        final_val = float(series.iloc[-1])
        for frac in [0.95, 0.97, 0.99]:
            row[f"{plabel}_{int(frac*100)}pct"] = _first_epoch(df, pcol, final_val * frac, "above")
    for col, thr, label, direction in [
        ("cka_to_final",             0.95, "cka_final_95", "above"),
        ("cka_to_final",             0.99, "cka_final_99", "above"),
        ("mean_future_cka",          0.95, "mfcka_95",     "above"),
        ("mean_future_cka",          0.99, "mfcka_99",     "above"),
        ("local_cka_change",         0.02, "local_cka_02", "below"),
        ("entk_distance_final",      0.05, "entk_dist_05", "below"),
        ("entk_distance_final",      0.01, "entk_dist_01", "below"),
        ("mean_future_entk_similarity", 0.95, "mfentk_95", "above"),
        ("mean_future_entk_similarity", 0.99, "mfentk_99", "above"),
    ]:
        row[label] = _first_epoch(df, col, thr, direction)
    rows.append(row)

if not rows:
    sat_df = pd.DataFrame(columns=["LR"])
    print("No data loaded; saturation table is empty.")
else:
    sat_df = pd.DataFrame(rows).set_index("LR")
    print("=== Saturation Epochs (first epoch reaching threshold) ===")
    print(sat_df.to_string())
    display(sat_df)


In [ ]:
# Step 7: Thesis hypothesis summary
# Central question: Does stabilization imply sufficiency?

def _first_ep(df, col, thr, direction):
    if col not in df.columns: return None
    for ep, v in zip(df["epoch"], df[col]):
        if pd.isna(v): continue
        if direction == "above" and v >= thr: return int(ep)
        if direction == "below" and v <= thr: return int(ep)
    return None

def _probe_sat(df, col, frac):
    if col not in df.columns: return None
    series = df[col].dropna()
    if series.empty: return None
    return _first_ep(df, col, float(series.iloc[-1]) * frac, "above")

thesis_rows = []
for lr in RUN_DIRS:
    df = dfs.get(lr, {}).get("master_trajectory.csv")
    if df is None: continue
    p95_log = _probe_sat(df, "logistic_acc", 0.95)
    p95_rbf = _probe_sat(df, "rbf_svm_acc",  0.95)
    p99_log = _probe_sat(df, "logistic_acc", 0.99)
    p99_rbf = _probe_sat(df, "rbf_svm_acc",  0.99)
    local_stab  = _first_ep(df, "local_cka_change",  0.02, "below")
    global_stab = _first_ep(df, "cka_to_final",      0.99, "above")
    entk_stab   = _first_ep(df, "entk_distance_final", 0.01, "below")

    def gap(stab, probe):
        if stab is None or probe is None: return "N/A"
        return f"{stab - probe:+d}"

    thesis_rows.append({
        "LR":                          lr,
        "probe95_logistic":            p95_log,
        "probe95_rbf_svm":             p95_rbf,
        "probe99_logistic":            p99_log,
        "probe99_rbf_svm":             p99_rbf,
        "local_stab (cka<=0.02)":      local_stab,
        "global_stab (cka>=0.99)":     global_stab,
        "entk_stab (dist<=0.01)":      entk_stab,
        "gap local-probe95_rbf":       gap(local_stab,  p95_rbf),
        "gap global-probe95_rbf":      gap(global_stab, p95_rbf),
    })

if not thesis_rows:
    thesis_df = pd.DataFrame(columns=["LR"])
    print("No data loaded; thesis table is empty.")
else:
    thesis_df = pd.DataFrame(thesis_rows).set_index("LR")
    print("=== Thesis Hypothesis Table ===")
    print("Positive gap => stabilization AFTER probe saturation (stab lags sufficiency)")
    print("Negative gap => stabilization BEFORE probe saturation (stab precedes sufficiency)\n")
    print(thesis_df.to_string())
    display(thesis_df)


In [ ]:
# Step 3 / Final Cell: Full inline HTML dashboard
# ─────────────────────────────────────────────────────────────────────────────
# The cell renders a scrollable dashboard directly in the notebook.
# No permanent HTML is written unless save_html=True.

def _img_b64(path):
    with open(path, "rb") as f:
        data = base64.b64encode(f.read()).decode()
    return f"data:image/png;base64,{data}"

def _html_table(df, caption=None, max_rows=None):
    if df is None or (hasattr(df, "__len__") and len(df) == 0):
        return "<p><em>No data</em></p>"
    if max_rows:
        df = df.head(max_rows)
    cap = f"<caption><b>{caption}</b></caption>" if caption else ""
    header = "".join(f"<th>{c}</th>" for c in df.columns)
    body = ""
    for _, row in df.iterrows():
        cells = ""
        for v in row:
            if pd.isna(v) if not isinstance(v, str) else False:
                cells += "<td style='color:#aaa'>NaN</td>"
            elif isinstance(v, (float, np.floating)):
                cells += f"<td>{v:.5g}</td>"
            else:
                cells += f"<td>{v}</td>"
        body += f"<tr>{cells}</tr>"
    return (
        "<table border=1 cellspacing=0 cellpadding=4 "
        "style='border-collapse:collapse;font-size:11px;font-family:monospace;margin:4px 0'>"
        f"{cap}<thead style='background:#f5f5f5'><tr>{header}</tr></thead>"
        f"<tbody>{body}</tbody></table>"
    )

def _final_metrics_table(df):
    WANT = [
        "epoch", "lr", "network_test_acc",
        "local_cka_change", "cka_to_final", "mean_future_cka",
        "nc1", "log10_nc1", "nc2_etf_deviation",
        "nc3_weight_mean_alignment", "nc4_ncm_disagreement",
        "entk_distance_prev", "entk_distance_final", "mean_future_entk_similarity",
        "logistic_acc", "linear_svm_acc", "rbf_svm_acc", "rf_acc",
    ]
    if df is None or df.empty: return "<p>No data</p>"
    avail = [c for c in WANT if c in df.columns]
    final = df[avail].iloc[-1]
    rows_html = "".join(
        f"<tr><td style='font-weight:bold;background:#f9f9f9'>{col}</td>"
        f"<td>{'N/A' if pd.isna(v) else (f'{v:.6g}' if isinstance(v,(float,np.floating)) else v)}</td></tr>"
        for col, v in final.items()
    )
    return (
        "<table border=1 cellspacing=0 cellpadding=4 "
        "style='border-collapse:collapse;font-size:11px;font-family:monospace'>"
        f"<thead><tr><th>Metric</th><th>Value (epoch 200)</th></tr></thead>"
        f"<tbody>{rows_html}</tbody></table>"
    )


def build_dashboard(save_html=False):
    'Render inline HTML dashboard. Pass save_html=True to also write .generated/dashboard.html.'
    SEC  = ("background:#1a1a2e;color:#e0e0e0;padding:10px 20px;margin:24px 0 6px 0;"
            "font-family:monospace;font-size:15px;letter-spacing:1px;border-left:4px solid #4fc3f7")
    SUB  = ("background:#f0f4ff;color:#1a237e;padding:5px 14px;margin:10px 0 4px 0;"
            "font-family:monospace;font-size:12px;border-left:3px solid #3949ab;font-weight:bold")
    META = "font-family:monospace;font-size:11px;color:#555;margin:2px 0"

    p = []   # html parts

    # ── Header ──────────────────────────────────────────────────────────────
    p.append(
        "<div style='font-family:monospace;background:#0d0d1a;padding:20px 28px;"
        "border-radius:6px;margin-bottom:16px'>"
        "<h1 style='color:#4fc3f7;margin:0;font-size:20px;letter-spacing:2px'>"
        "FINAL THESIS RESULTS DASHBOARD</h1>"
        "<p style='color:#78909c;margin:6px 0 0 0;font-size:11px'>"
        "ResNet-18 &nbsp;·&nbsp; CIFAR-10 &nbsp;·&nbsp; SGD + CosineAnnealing"
        " &nbsp;·&nbsp; 200 epochs &nbsp;·&nbsp; seed=42<br>"
        "<b style='color:#b0bec5'>Question:</b> "
        "Does representation stabilization imply representation sufficiency?</p></div>"
    )

    # ── Cross-LR overview ────────────────────────────────────────────────────
    p.append(f"<div style='{SEC}'>CROSS-LEARNING-RATE COMPARISON</div>")
    ov_rows = []
    for lr in RUN_DIRS:
        df = dfs.get(lr, {}).get("master_trajectory.csv")
        if df is None: continue
        final = df.iloc[-1]
        def g(c): return round(float(final[c]), 5) if c in final.index and not pd.isna(final[c]) else "—"
        ov_rows.append({"LR": lr,
            "test_acc":        g("network_test_acc"),
            "logistic":        g("logistic_acc"),
            "rbf_svm":         g("rbf_svm_acc"),
            "cka_local":       g("local_cka_change"),
            "cka_to_final":    g("cka_to_final"),
            "log10_nc1":       g("log10_nc1"),
            "nc2_etf_dev":     g("nc2_etf_deviation"),
            "entk_dist_final": g("entk_distance_final"),
        })
    p.append(_html_table(pd.DataFrame(ov_rows), "Final-Epoch Summary Across All Learning Rates"))

    # ── Per-LR sections ──────────────────────────────────────────────────────
    for lr, info in runs.items():
        color = LR_COLORS[lr]
        p.append(
            f"<div style='{SEC};border-left-color:{color}'>"
            f"LR = {lr} &nbsp;&#183;&nbsp; {info['dir']}</div>"
        )
        p.append(f"<p style='{META}'><b>Path:</b> {info['path']}</p>")
        p.append(f"<p style='{META}'><b>Exists:</b> {info['exists']}</p>")

        if not info["exists"]:
            p.append("<p style='color:#f44336'><b>Run directory not found.</b></p>")
            continue

        # available files
        p.append(f"<div style='{SUB}'>Available Files</div>")
        p.append("<ul style='font-size:11px;font-family:monospace;color:#444;margin:4px 0 8px 20px'>")
        for fn in sorted(info["csvs"]): p.append(f"<li>results/{fn}</li>")
        for jp in info["jsons"]: p.append(f"<li>results/{os.path.basename(jp)}</li>")
        for pp_ in info["plots"]: p.append(f"<li>plots/{os.path.basename(pp_)}</li>")
        p.append("</ul>")

        # final metrics
        p.append(f"<div style='{SUB}'>Final-Epoch Metrics (epoch 200)</div>")
        df_m = dfs.get(lr, {}).get("master_trajectory.csv")
        p.append(_final_metrics_table(df_m))

        # pre-generated plots
        if info["plots"]:
            p.append(f"<div style='{SUB}'>Pre-generated Plots</div>")
            p.append("<div style='display:flex;flex-wrap:wrap;gap:10px;padding:6px 0'>")
            for pp_ in sorted(info["plots"]):
                try:
                    b64  = _img_b64(pp_)
                    name = os.path.basename(pp_).replace(".png","").replace("_"," ")
                    p.append(
                        f"<figure style='margin:0;text-align:center'>"
                        f"<img src='{b64}' style='height:200px;border:1px solid #ddd;border-radius:4px'>"
                        f"<figcaption style='font-size:10px;color:#777;margin-top:2px'>{name}</figcaption>"
                        f"</figure>"
                    )
                except Exception as exc:
                    p.append(f"<p style='color:red'>Could not load {os.path.basename(pp_)}: {exc}</p>")
            p.append("</div>")

        # CSV previews (collapsible)
        p.append(f"<div style='{SUB}'>CSV Previews</div>")
        for csv_name in CSV_NAMES:
            df_c = dfs.get(lr, {}).get(csv_name)
            if df_c is None:
                p.append(f"<p style='color:#f44336;font-size:11px'>{csv_name}: not loaded</p>")
                continue
            p.append(
                f"<details style='margin:5px 0'>"
                f"<summary style='cursor:pointer;font-size:12px;font-weight:bold;color:#1565c0'>"
                f"{csv_name} "
                f"<span style='color:#777;font-weight:normal'>({df_c.shape[0]} rows x {df_c.shape[1]} cols)</span>"
                f"</summary>"
            )
            p.append(_html_table(df_c.head(3), f"First 3 rows — {csv_name}"))
            p.append(_html_table(df_c.tail(2), "Last 2 rows"))
            p.append("</details>")

    # ── Saturation analysis ──────────────────────────────────────────────────
    p.append(f"<div style='{SEC}'>SATURATION ANALYSIS</div>")
    p.append("<p style='font-size:12px;color:#555'>First epoch reaching each threshold "
             "(None = threshold never reached).</p>")
    p.append(_html_table(sat_df.reset_index(), "Saturation Epochs"))

    # ── Thesis hypothesis ────────────────────────────────────────────────────
    p.append(f"<div style='{SEC}'>THESIS HYPOTHESIS SUMMARY</div>")
    p.append(
        "<p style='font-size:12px;color:#555'>"
        "<b>Positive gap (stab - probe95):</b> stabilization AFTER probe saturation "
        "&#8594; stab is a <em>lagging</em> indicator of sufficiency.<br>"
        "<b>Negative gap:</b> stabilization BEFORE probe saturation "
        "&#8594; stab <em>precedes</em> sufficiency.<br>"
        "<b>Near-zero gap:</b> simultaneous emergence.</p>"
    )
    p.append(_html_table(thesis_df.reset_index(), "Thesis Summary Table"))

    # ── Footer ───────────────────────────────────────────────────────────────
    p.append(
        "<div style='font-size:10px;color:#90a4ae;margin-top:28px;"
        "border-top:1px solid #eee;padding-top:8px;font-family:monospace'>"
        "Rendered inline — no permanent HTML written unless save_html=True."
        "</div>"
    )

    full_html = "\n".join(p)
    wrapper = (
        "<div style='max-height:90vh;overflow-y:auto;overflow-x:auto;"
        "padding:16px;border:1px solid #e0e0e0;border-radius:6px;"
        "background:#fff;font-family:sans-serif'>"
        + full_html + "</div>"
    )

    if save_html:
        out_path = os.path.join(GEN_DIR, "dashboard.html")
        with open(out_path, "w", encoding="utf-8") as fh:
            fh.write(f"<!DOCTYPE html><html><head><meta charset='utf-8'></head><body>{full_html}</body></html>")
        print(f"Saved: {out_path}")

    display(HTML(wrapper))


build_dashboard(save_html=False)
